In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_validate
from sklearn.model_selection import ShuffleSplit
from sklearn import metrics
from numpy import mean
from numpy import std
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
df_1= pd.read_csv('dataset.csv') #Import the inhibitor dataset
df_1 = df_1.dropna()
df_clean = df_1.set_index('compound')
df_clean

In [ ]:
def categorize_ic50(ic50):
    if ic50 <= 1:
        return 'Strong'
    else:
        return 'Weak' #Define the inhibition potency

df_1['Inhibitor_Category'] = df_1['IC50'].apply(categorize_ic50)

In [ ]:
features = df_clean.loc[:,'MaxEStateIndex':'fr_urea']
x= np.array(features)
scaler = StandardScaler()
scaler.fit(x)
x = scaler.transform(x) #Feature Extraction and Data Standardization

feature_names = [f"Feature {i}" for i in range(x.shape[1])] #Change Feature name to order

In [ ]:
df_1['Inhibitor_Category'].astype('category')
labelencoder = LabelEncoder()
y= labelencoder.fit_transform(df_1['Inhibitor_Category'])
y = np.array(y)
# 0 is potent, 1 is non-potent

In [ ]:
# PCA

In [ ]:
pcamodel = PCA(random_state = 0)
transformed_features = pcamodel.fit_transform(x)
x_component, y_component, = transformed_features[:, 0], transformed_features[:, 1]
plt.figure
plt.figure(dpi=130)
scatter = plt.scatter(x=x_component, y=y_component, c=y, marker='.', cmap='Set1')
plt.xlabel('PC1')
plt.ylabel('PC2')
unique_classes = np.unique(y)
custom_labels = ['Potent', 'Less Potent']  # Replace with actual labels
handles = scatter.legend_elements()[0]
plt.legend(handles, custom_labels,loc="upper left", title='bGUS_Inhibitors')
plt.savefig("PCA fit.png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Baseline Model

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate, ShuffleSplit

# Definition
cv = ShuffleSplit(n_splits=3, test_size=0.2, random_state=0)

# Build Dummy Classifier
dummy = DummyClassifier(strategy='most_frequent', random_state=0)

scores = cross_validate(dummy, x, y, cv=cv,
                        scoring=('accuracy','roc_auc', 'balanced_accuracy',
                                 'recall_weighted', 'precision_weighted','f1_weighted'),
                        return_train_score=True)

# Performance metrics
score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
# Extra Tree

In [ ]:
cv = ShuffleSplit(n_splits=3, test_size=0.2, random_state=0)

etc = ExtraTreesClassifier(random_state=0)

scores = cross_validate(etc, x, y, cv=cv, scoring=('accuracy','roc_auc', 'balanced_accuracy', 'recall_weighted', 'precision_weighted','f1_weighted'),
                        return_train_score=True)

score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores ['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
#Perform grid search to optimise hyperparameters
et_random = ExtraTreesClassifier(random_state=0)
# Number of trees
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]
# Number of features to consider at every split
max_features = ['auto', 'sqrt', 'log2']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 110, num = 11)]
max_depth.append(None)
# Minimum number of samples required to split a node
min_samples_split = [2,4,6,8,10]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1,2,3,4]
# Method of selecting samples for training each tree
bootstrap = [True, False]
# Weights associated with classes
class_weight = ['balanced', 'balanced_subsample']
# Create the random grid
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap,
               'class_weight': class_weight}

# Random search across 100 different combinations
et_random = RandomizedSearchCV(estimator = et_random, param_distributions = random_grid,
                               n_iter = 50, cv = 3, verbose=2, random_state=0, n_jobs = -1)

# Fit the model
et_random.fit(x, y)
print(et_random.best_params_)

etc = ExtraTreesClassifier(random_state=0, n_estimators=1400, min_samples_split = 2, min_samples_leaf=1, max_features= 'auto', max_depth=80,
                           class_weight='balanced', bootstrap= False)

scores = cross_validate(etc, x, y, cv=cv, scoring=('accuracy','roc_auc', 'balanced_accuracy', 'recall_weighted', 'precision_weighted','f1_weighted'),
                        return_train_score=True)

score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
cv = ShuffleSplit(n_splits= 3, test_size=0.2, random_state=0)

rf = RandomForestClassifier(random_state=0)

scores = cross_validate(rf, x, y, cv=cv, scoring=('accuracy', 'roc_auc', 'balanced_accuracy', 'recall_weighted', 'precision_weighted','f1_weighted'),
                        return_train_score=True)

score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
#Perform grid search to optimise hyperparameters
from sklearn.model_selection import RandomizedSearchCV
rf_random = RandomForestClassifier(random_state=0)
# Number of trees
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]
# criterion
criterion = ['gini', 'entropy','log_loss']
# Number of features to consider at every split
max_features = ['auto','sqrt', 'log2']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 110, num = 11)]
max_depth.append(None)
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]
# Method of selecting samples for training each tree
bootstrap = [True, False]
# Create the random grid
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap,
               'criterion': criterion}

# Random search across 100 different combinations
rf_random = RandomizedSearchCV(estimator = rf_random, param_distributions = random_grid,
                               n_iter = 50, cv = 3, verbose=2, random_state=0, n_jobs = -1)
#Fit the model
rf_random.fit(x, y)
print(rf_random.best_params_)

rtf = RandomForestClassifier(random_state=0, n_estimators=1400, min_samples_split = 2, min_samples_leaf=1, max_features= 'auto', max_depth=None,
                           criterion ='entropy', bootstrap= True)

scores = cross_validate(
    rtf, x, y, cv=cv,
    scoring=('accuracy', 'roc_auc', 'balanced_accuracy', 'recall_weighted', 'precision_weighted', 'f1_weighted'),
    return_train_score=True
)


score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score_roc = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score_roc.mean(), score_roc.std()))

score_recall = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score_recall.mean(), score_recall.std()))

score_precision = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score_precision.mean(), score_precision.std()))

score_balanced = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score_balanced.mean(), score_balanced.std()))

print("%0.3f f1_weighted ± %0.3f" % (scores['test_f1_weighted'].mean(), scores['test_f1_weighted'].std()))

In [ ]:
rtf.fit(x, y)
importances = rtf.feature_importances_
indices = np.argsort(importances)[::-1]

# Plot top 20 most important features
plt.figure(figsize=(10, 6))
plt.title("Top 20 Feature Importances Random Forest")
plt.bar(range(20), importances[indices[:20]], align='center', color='pink')
plt.xticks(range(20), [feature_names[i] for i in indices[:20]], rotation=45)
plt.ylabel("Importance")
plt.tight_layout()
plt.figure(figsize=(10, 6))
plt.title("Top 20 Feature Importances")
plt.bar(range(20), importances[indices[:20]], align='center', color='pink')
plt.xticks(range(20), [feature_names[i] for i in indices[:20]], rotation=45)
plt.ylabel("Importance")
plt.tight_layout()

plt.show()

In [ ]:
# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
cv = ShuffleSplit(n_splits= 3, test_size=0.2, random_state=0)

dct = DecisionTreeClassifier(random_state=0)

scores = cross_validate(dct, x, y, cv=cv, scoring=('accuracy','roc_auc', 'balanced_accuracy', 'recall_weighted', 'precision_weighted','f1_weighted'),
                        return_train_score=True)

score_acc_1 = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc_1.mean(), score_acc_1.std()))

score1_dt = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1_dt.mean(), score1_dt.std()))

score2_dt = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2_dt.mean(), score2_dt.std()))

score3_dt = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3_dt.mean(), score3_dt.std()))

score4_dt = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4_dt.mean(), score4_dt.std()))

score5_dt = scores['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5_dt.mean(), score5_dt.std()))

In [ ]:
from sklearn.model_selection import GridSearchCV, ShuffleSplit
from sklearn.tree import DecisionTreeClassifier

# Parameter grid to search
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 5, 10],
    'max_features': [None, 'sqrt', 'log2'],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(
    estimator=dct,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='f1_weighted',  
    n_jobs=-1,               
    verbose=1       
)

grid_search.fit(x, y)

# Best hyperparameters
print("Best Parameters:", grid_search.best_params_)

# Best model performance
print("Best Weighted F1 Score: %0.3f" % grid_search.best_score_)

dct2 = DecisionTreeClassifier(criterion = 'entropy', max_depth=None, max_features='log2', min_samples_leaf=1,
                             min_samples_split= 5, random_state=0)

scores = cross_validate(dct2, x, y, cv=cv, scoring=('accuracy','roc_auc', 'balanced_accuracy', 'recall_weighted', 'precision_weighted','f1_weighted'),
                        return_train_score=True)

score_acc_1 = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc_1.mean(), score_acc_1.std()))

score1_dt = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1_dt.mean(), score1_dt.std()))

score2_dt = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2_dt.mean(), score2_dt.std()))

score3_dt = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3_dt.mean(), score3_dt.std()))

score4_dt = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4_dt.mean(), score4_dt.std()))

score5_dt = scores['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5_dt.mean(), score5_dt.std()))

In [ ]:
# kNN-1,2,4

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=1)

scores = cross_validate(knn, x, y, cv=cv, scoring=('accuracy','roc_auc', 'balanced_accuracy', 'recall_weighted', 'precision_weighted','f1_weighted'),
                        return_train_score=True)

score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
kNN_2 = KNeighborsClassifier (n_neighbors = 2)

scores = cross_validate(kNN_2, x, y, cv=cv, scoring=('accuracy','roc_auc', 'balanced_accuracy', 'recall_weighted', 'precision_weighted','f1_weighted'),
                        return_train_score=True)

score_acc_kNN = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc_kNN.mean(), score_acc_kNN.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores['test_f1_weighted']
print("%0.3f test_f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
kNN_4 = KNeighborsClassifier (n_neighbors = 4)

scores = cross_validate(kNN_4, x, y, cv=cv, scoring=('accuracy','roc_auc', 'balanced_accuracy', 'recall_weighted', 'precision_weighted','f1_weighted'),
                        return_train_score=True)

score_acc_kNN = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc_kNN.mean(), score_acc_kNN.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores['test_f1_weighted']
print("%0.3f test_f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
# LinearSVC

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_validate, ShuffleSplit

svm = LinearSVC(
    random_state=0,
    max_iter=10000,  
    dual=False        
)

# Evaluation
scores = cross_validate(svm, x, y, cv=cv, 
                        scoring=('accuracy','roc_auc', 'balanced_accuracy', 
                                 'recall_weighted', 'precision_weighted','f1_weighted'),
                        return_train_score=True)

score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
# Linear Lasso

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, ShuffleSplit

# Logistic Regression with Lasso
lasso_lr = LogisticRegression(
    penalty='l1',        
    solver='liblinear',  
    random_state=0,      
    max_iter=1000,       
)

# Evaluation
scores = cross_validate(lasso_lr, x, y, cv=cv, 
                        scoring=('accuracy', 'roc_auc', 'balanced_accuracy', 
                                 'recall_weighted', 'precision_weighted', 'f1_weighted'),
                        return_train_score=True)

score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
# Linear Ridge

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, ShuffleSplit

# Logistic Regression with Ridge 
ridge_lr = LogisticRegression(
    penalty='l2',        
    solver='lbfgs',     
    random_state=0,      
    max_iter=1000,       
)

# Evaluation
scores = cross_validate(ridge_lr, x, y, cv=cv, 
                        scoring=('accuracy', 'roc_auc', 'balanced_accuracy', 
                                'recall_weighted', 'precision_weighted', 'f1_weighted'),
                        return_train_score=True)

score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
# XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate, ShuffleSplit

# Build XGBoost model with default hyperparameters
xgb = XGBClassifier(random_state=0)

# Evaluation
scores = cross_validate(xgb, x, y, cv=cv, 
                        scoring=('accuracy', 'roc_auc', 'balanced_accuracy', 
                                'recall_weighted', 'precision_weighted', 'f1_weighted'),
                        return_train_score=True)

score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
# Define scoring metrics
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'roc_auc': make_scorer(roc_auc_score),
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'recall_weighted': make_scorer(recall_score, average='weighted'),
    'precision_weighted': make_scorer(precision_score, average='weighted'),
    'f1_weighted': make_scorer(f1_score, average='weighted')
}

# Define the parameter grid to search
param_grid = {
    'learning_rate': [0.01, 0.1, 0.3],      # Step size shrinkage
    'max_depth': [3, 5, 7],                  # Maximum tree depth
    'min_child_weight': [1, 3, 5],           # Minimum sum of instance weight needed in a child
    'subsample': [0.8, 1.0],                 # Subsample ratio of training instances
    'colsample_bytree': [0.8, 1.0],          # Subsample ratio of columns
    'n_estimators': [100, 200],              # Number of trees
    'gamma': [0, 0.1, 0.2]                   # Minimum loss reduction required
}

# XGBoost
xgb = XGBClassifier(
    random_state=0,
    eval_metric='logloss',
    use_label_encoder=False,
    n_jobs=-1  # Use all available cores
)

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring=scoring,
    refit='roc_auc',  # Optimise for AUROC, but keep all scores
    cv=cv,
    verbose=2,
    n_jobs=-1  
)

grid_search.fit(x, y)

# Print best parameters and scores
print("Best parameters found: ", grid_search.best_params_)
print("Best AUROC score: ", grid_search.best_score_)

# Get all cross-validation results
cv_results = grid_search.cv_results_
for metric in scoring.keys():
    print(f"\nBest {metric}: {cv_results[f'mean_test_{metric}'][grid_search.best_index_]:.3f} (±{cv_results[f'std_test_{metric}'][grid_search.best_index_]:.3f})")

# Access the best model
best_xgb = grid_search.best_estimator_

In [ ]:
import matplotlib.pyplot as plt
from xgboost import plot_importance

# Get feature importance scores
importance_scores = best_xgb.get_booster().get_score(importance_type='weight')

# Sort features by importance
sorted_features = sorted(importance_scores.items(), key=lambda x: x[1], reverse=True)
top_20 = sorted_features[:20]

# Print top 20 features
print("Top 20 Important Features:")
for i, (feature, score) in enumerate(top_20, 1):
    print(f"{i}. {feature}: {score:.4f}")

# Plot feature importance
plt.figure(figsize=(12, 8))
plot_importance(best_xgb, max_num_features=20, importance_type='weight')
plt.title('Top 20 Feature Importance (Weight)')

plt.show()

In [ ]:
# Models with Selective Features:

In [ ]:
RF_features = df_clean.loc[:,['MaxEStateIndex', 'SMR_VSA10', 'MaxAbsEStateIndex', 'VSA_EState4', 'BCUT2D_MRLOW', 
                              'fr_bicyclic', 'SlogP_VSA9', 'MaxPartialCharge', 'MinAbsPartialCharge', 'VSA_EState10', 
                              'VSA_EState5', 'SlogP_VSA7', 'MolLogP', 'VSA_EState2', 'Kappa1', 'qed', 'fr_NH1', 
                              'FpDensityMorgan3', 'MinAbsEStateIndex', 'NumHAcceptors']]

XGB_features = df_clean.loc[:,['SMR_VSA10', 'VSA_EState5', 'qed', 'MaxEStateIndex', 'Kappa3', 'VSA_EState4', 'SlogP_VSA1', 
                               'fr_bicyclic', 'BCUT2D_MRLOW', 'VSA_EState2', 'MinAbsEStateIndex', 'BCUT2D_LOGPLOW', 'EState_VSA9', 
                               'fr_Ar_OH', 'BCUT2D_CHGHI', 'SlogP_VSA8', 'NumHAcceptors', 'NumHeteroatoms', 
                               'Kappa1', 'MaxPartialCharge']]

x= np.array(RF_features)
scaler = StandardScaler()
scaler.fit(x)
x = scaler.transform(x)

feature_names = [f"Feature {i}" for i in range(x.shape[1])]

In [ ]:
# Random Forest model with seletive features

In [ ]:
rf_sel = RandomForestClassifier(random_state=0, n_estimators=1400, min_samples_split = 2, min_samples_leaf=1, max_features= 'auto', max_depth=None,
                           criterion ='entropy', bootstrap= True)
#Outline scoring method and verify performance
scores = cross_validate(
    rf_sel, x, y, cv=cv,
    scoring=('accuracy', 'roc_auc', 'balanced_accuracy', 'recall_weighted', 'precision_weighted', 'f1_weighted'),
    return_train_score=True
)

score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score_roc = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score_roc.mean(), score_roc.std()))

score_recall = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score_recall.mean(), score_recall.std()))

score_precision = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score_precision.mean(), score_precision.std()))

score_balanced = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score_balanced.mean(), score_balanced.std()))

print("%0.3f f1_weighted ± %0.3f" % (scores['test_f1_weighted'].mean(), scores['test_f1_weighted'].std()))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)

rf_sel.fit(X_train, y_train)

# Generate predictions
y_pred = rf_sel.predict(X_test)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualize with sklearn's built-in display
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                             display_labels=['potent', 'less-potent'])
disp.plot(cmap='Blues')
plt.title('Random Forest Confusion Matrix')

plt.savefig("RF_ConfusionMatrix.jpeg", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# XGBoost with selective features

In [ ]:
from xgboost import XGBClassifier

# Build XGBoost model with selected hyperparameters
xgb_sel = XGBClassifier(random_state=0, colsample_bytree = 1.0, gamma = 0.2, learning_rate = 0.1, max_depth = 3, 
                    min_child_weight = 1, n_estimators = 100, subsample = 1.0)

# Evaluation
scores = cross_validate(xgb_sel, x, y, cv=cv, 
                        scoring=('accuracy', 'roc_auc', 'balanced_accuracy', 
                                'recall_weighted', 'precision_weighted', 'f1_weighted'),
                        return_train_score=True)

score_acc = scores['test_accuracy']
print("%0.3f accuracy with a standard deviation of %0.3f" % (score_acc.mean(), score_acc.std()))

score1 = scores['test_roc_auc']
print("%0.3f AUROC with a standard deviation of %0.3f" % (score1.mean(), score1.std()))

score2 = scores['test_recall_weighted']
print("%0.3f recall_weighted with a standard deviation of %0.3f" % (score2.mean(), score2.std()))

score3 = scores['test_precision_weighted']
print("%0.3f precision_weighted with a standard deviation of %0.3f" % (score3.mean(), score3.std()))

score4 = scores['test_balanced_accuracy']
print("%0.3f balanced accuracy with a standard deviation of %0.3f" % (score4.mean(), score4.std()))

score5 = scores['test_f1_weighted']
print("%0.3f f1_weighted with a standard deviation of %0.3f" % (score5.mean(), score5.std()))

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)

xgb_sel.fit(X_train, y_train)

# Generate predictions
y_pred = xgb_sel.predict(X_test)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualise with sklearn's built-in display
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                             display_labels=['potent', 'non-potent'])
disp.plot(cmap='Reds')
plt.title('XGBoost Confusion Matrix')

plt.show()